# Perfilamiento reproducible · Spotify y Grammy
Requerimientos AR1–AR3: cobertura de entradas Grammy, popularidad y energía del snapshot Spotify. Consultar `docs/requirements.md`. Este notebook recolecta evidencia sin modificar fuentes. Grammy se consulta desde PostgreSQL, Spotify desde el CSV original. Ejecutar después de preparar `source.grammy`.
La columna `winner` no se acepta como clasificación verificada y las fechas de publicación web no representan fechas musicales.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
from sqlalchemy import text
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root))
from scripts.local_env import configure
from src.common import engine
from src.profiling import profile_sources, CATEGORIES
configure()
spotify = pd.read_csv(root / 'data/input/spotify_dataset.csv')
with engine().connect() as conn:
    grammy_source = pd.read_sql(text('SELECT * FROM source.grammy ORDER BY source_row_id'), conn)
# Vista de análisis: vacíos como ausencias y tipos interpretados; source.* permanece intacta.
grammy = grammy_source.drop(columns='source_row_id').replace('', pd.NA).copy()
grammy['year'] = pd.to_numeric(grammy['year'])
grammy['winner'] = grammy['winner'].map({'True': True, 'False': False})
profile = profile_sources(spotify, grammy)
print('Spotify:', len(spotify), 'Grammy PostgreSQL:', len(grammy_source))

Spotify: 114000 Grammy PostgreSQL: 4810


## Estructura, completitud y cardinalidades
Las ausencias se interpretan por contexto: `artist` puede faltar en categorías fuera de alcance, pero título y artista son necesarios para el enlace de las categorías seleccionadas.

In [2]:
for name, frame in [('Spotify', spotify), ('Grammy', grammy)]:
    print(name)
    display(pd.DataFrame({'dtype': frame.dtypes.astype(str), 'missing': frame.isna().sum(), 'missing_pct': frame.isna().mean()*100, 'distinct': frame.nunique()}))
print('Filas duplicadas sin índice exportado:', profile['spotify']['duplicates_without_export_index'])
print('Repeticiones track_id:', profile['spotify']['duplicate_track_ids'])
print('IDs con conflicto en atributos analíticos:', profile['spotify']['conflicting_track_ids'])

Spotify


,dtype,missing,missing_pct,distinct
Unnamed: 0,int64,0,0.000000,114000
track_id,object,0,0.000000,89741
artists,object,1,0.000877,31437
album_name,object,1,0.000877,46589
track_name,object,1,0.000877,73608
popularity,int64,0,0.000000,101
duration_ms,int64,0,0.000000,50697
explicit,bool,0,0.000000,2
danceability,float64,0,0.000000,1174
energy,float64,0,0.000000,2083


Grammy


,dtype,missing,missing_pct,distinct
year,int64,0,0.000000,62
title,object,0,0.000000,62
published_at,object,0,0.000000,4
updated_at,object,0,0.000000,10
category,object,0,0.000000,638
nominee,object,6,0.124740,4131
artist,object,1840,38.253638,1658
workers,object,2190,45.530146,2366
img,object,1367,28.419958,1463
winner,bool,0,0.000000,1


Filas duplicadas sin índice exportado: 450
Repeticiones track_id: 24259
IDs con conflicto en atributos analíticos: 720


## Distribuciones numéricas y dominios
Popularidad: 0–100; energía y bailabilidad: 0–1, según el contrato de atributos Spotify. Estos límites se justifican por las unidades del indicador, no por adaptar el umbral al lote.

In [3]:
display(spotify.describe().T)
display(pd.DataFrame([{'attribute': col, 'invalid_range': int((~spotify[col].between(0, maximum)).sum())} for col, maximum in [('popularity',100),('energy',1),('danceability',1)]]))
for col in ['track_genre', 'artists', 'explicit']:
    display(spotify[col].value_counts(dropna=False).head(15).to_frame('count'))
display(grammy.category.value_counts().head(20).to_frame('count'))
display(grammy.winner.value_counts(dropna=False).to_frame('count'))

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,114000.0,56999.500000,32909.109681,0.000,28499.75000,56999.500000,85499.2500,113999.000
popularity,114000.0,33.238535,22.305078,0.000,17.00000,35.000000,50.0000,100.000
duration_ms,114000.0,228029.153114,107297.712645,0.000,174066.00000,212906.000000,261506.0000,5237295.000
danceability,114000.0,0.566800,0.173542,0.000,0.45600,0.580000,0.6950,0.985
energy,114000.0,0.641383,0.251529,0.000,0.47200,0.685000,0.8540,1.000
key,114000.0,5.309140,3.559987,0.000,2.00000,5.000000,8.0000,11.000
loudness,114000.0,-8.258960,5.029337,-49.531,-10.01300,-7.004000,-5.0030,4.532
mode,114000.0,0.637553,0.480709,0.000,0.00000,1.000000,1.0000,1.000
speechiness,114000.0,0.084652,0.105732,0.000,0.03590,0.048900,0.0845,0.965
acousticness,114000.0,0.314910,0.332523,0.000,0.01690,0.169000,0.5980,0.996


,attribute,invalid_range
0,popularity,0
1,energy,0
2,danceability,0


,count
track_genre,
acoustic,1000
afrobeat,1000
alt-rock,1000
alternative,1000
ambient,1000
anime,1000
black-metal,1000
bluegrass,1000
blues,1000


,count
artists,
The Beatles,279
George Jones,271
Stevie Wonder,236
Linkin Park,224
Ella Fitzgerald,222
Prateek Kuhad,217
Feid,202
Chuck Berry,190
Håkan Hellström,183


,count
explicit,
False,104253
True,9747


,count
category,
Song Of The Year,70
Record Of The Year,69
Album Of The Year,66
Best Opera Recording,64
Best Album Notes,63
Best Country Song,55
Best Instrumental Composition,55
Best New Artist,51
Best Historical Album,44


,count
winner,
True,4810


## Temporalidad y alcance
Año fuente conservado literalmente. No se reinterpreta 2019 como 2020 aunque la ceremonia se publique en 2020. No hay fecha de snapshot Spotify: no se puede medir evolución histórica de popularidad.

In [4]:
display(grammy.groupby('year').size().to_frame('entries'))
print('Huecos de año:', sorted(set(range(int(grammy.year.min()), int(grammy.year.max())+1))-set(grammy.year)))
display(pd.DataFrame({c: profile['grammy'][c+'_parse'] for c in ['published_at','updated_at']}))
scope = grammy[grammy.category.isin(CATEGORIES)]
display(scope.groupby('category').agg(entries=('nominee','size'), missing_artist=('artist',lambda x: x.isna().sum())))
display(scope.groupby(['year','category']).winner.sum().loc[lambda x: x>1].to_frame('true_flags'))

,entries
year,
1958,28
1959,35
1960,39
1961,41
1962,39
...,...
2015,84
2016,85
2017,86


Huecos de año: []


,published_at,updated_at
invalid_or_missing,0,0
min,2017-11-28 08:03:45+00:00,2017-11-28 08:03:45+00:00
max,2020-05-19 12:10:28+00:00,2020-09-01 19:16:40+00:00


,entries,missing_artist
category,,
Best Pop Solo Performance,13,0
Record Of The Year,69,0


true_flags
year category                             
2019 Best Pop Solo Performance           5
     Record Of The Year                  8

## Compatibilidad entre fuentes
No hay clave común global. Se exploran título y artista con normalización NFKC, casefold y espacios. Spotify separa colaboradores por `;`. No se divide el crédito Grammy por `&` ni se eliminan versiones/remasterizaciones: eso podría inventar identidades. El número de candidatos se calcula sobre IDs distintos; los conflictos por ID necesitan tratamiento explícito.

In [5]:
from collections import defaultdict
from src.transform import normalize
keys = defaultdict(set)
for row in spotify[['track_id','track_name','artists']].dropna().itertuples(index=False):
    for artist in row.artists.split(';'):
        keys[(normalize(row.track_name),normalize(artist))].add(row.track_id)
candidate_counts = scope.apply(lambda r: len(keys[(normalize(r.nominee),normalize(r.artist))]),axis=1)
display(candidate_counts.value_counts().sort_index().to_frame('entries_per_candidate_count'))
print('Créditos Spotify con colaboradores:', int(spotify.artists.fillna('').str.contains(';',regex=False).sum()))
print('Variantes de título por casefold:', spotify.track_name.nunique()-spotify.track_name.map(normalize).nunique())
print('El enlace definitivo excluirá IDs conflictivos; esta tabla es exploración, no carga del DW.')

,entries_per_candidate_count
0,60
1,13
2,5
3,2
7,2


Créditos Spotify con colaboradores: 30075
Variantes de título por casefold: 1214
El enlace definitivo excluirá IDs conflictivos; esta tabla es exploración, no carga del DW.


## Riesgos y decisiones
| Evidencia | Riesgo | Decisión / reglas | Requerimiento |
|---|---|---|---|
| IDs repetidos y conflictos Spotify | Multiplicación o elección arbitraria de medidas | S03; colapsar repeticiones compatibles, cuarentena de conflictos | AR1–AR3 |
| Ausencias de nombres | Falsos enlaces por claves vacías | S04/G05; cuarentena Spotify, bloqueo de identidad Grammy en alcance | AR1 |
| winner constante | Interpretar incorrectamente los resultados del premio | G06 Warning; no usar como KPI | AR1–AR3 |
| Distintas versiones por título/artista | Uniones many-to-many | P07, estados ambiguous/unmatched explícitos | AR1–AR3 |
| Rangos y unidades | Promedios imposibles | S05 y P08 Critical | AR2–AR3 |
| Fechas web y ausencia de snapshot | Afirmaciones temporales sin respaldo | año literal y limitación explícita | AR1–AR3 |

Los hallazgos completos y resultados operativos están en `docs/evidence`. Las reglas Critical usan 100% de conformidad; las Warning registran cada desviación sin esconderla.